In [19]:
import pandas as pd

## 1. Read CSV and set ID as index

In [56]:
df = pd.read_csv(
    "data/auto.csv",
    index_col="ID"
)

df.head()
df.index.name
df.columns

Index(['CarNumber', 'Make_n_model', 'Refund', 'Fines', 'History'], dtype='object')

## 2. Count number of observations

In [57]:
counts_per_column = df.count()
counts_per_column

CarNumber       931
Make_n_model    931
Refund          914
Fines           869
History          82
dtype: int64

## For total rows !!

In [58]:
rows = df.shape[0]
rows

931

## shown colums 

In [59]:
df.columns

Index(['CarNumber', 'Make_n_model', 'Refund', 'Fines', 'History'], dtype='object')

## 3. Drop duplicates and compare observation counts

In [60]:
# use exist colum 
rows_before = df["CarNumber"].count()
rows_before

931

In [61]:
# Before Delete 
rows_before = df["CarNumber"].count()

# Delete duplicates
df = df.drop_duplicates(
    subset=["CarNumber", "Make_n_model", "Fines"],
    keep="last"
)

# after delete 
rows_after = df["CarNumber"].count()
removed = rows_before - rows_after
#print 
rows_before, rows_after, removed

(931, 725, 206)

## 4. Missing values: check, drop columns, and fill

In [62]:
# The number of NaN before any processing
missing_before = df.isna().sum()
missing_before

CarNumber         0
Make_n_model      0
Refund           12
Fines            60
History         660
dtype: int64

In [63]:
# Number of rows
n_rows = len(df)

# Any column with more than 500 NaN should be dropped.
# non-missing = n_rows - missing
# condition missing > 500  <=>  non-missing < n_rows - 500
thresh = n_rows - 500

# Drop columns with more than 500 missing values
df = df.dropna(axis=1, thresh=thresh)

missing_after_drop_cols = df.isna().sum()
missing_after_drop_cols

CarNumber        0
Make_n_model     0
Refund          12
Fines           60
dtype: int64

## Fill missing values in Refund with previous value

In [64]:
# Compute mean of Fines (ignores NaN by default)
mean_fines = df["Fines"].mean()

# Fill missing Fines with the mean
df["Fines"] = df["Fines"].fillna(mean_fines)

# Final check for NaN values
missing_final = df.isna().sum()
missing_final

CarNumber        0
Make_n_model     0
Refund          12
Fines            0
dtype: int64

In [65]:
# Fill missing values in Refund with previous non-missing value (forward fill)
df["Refund"] = df["Refund"].ffill()

missing_after_refund = df.isna().sum()
missing_after_refund

CarNumber       0
Make_n_model    0
Refund          0
Fines           0
dtype: int64

In [54]:
mean_fines = df["Fines"].mean()
df["Fines"] = df["Fines"].fillna(mean_fines)
missing_final = df.isna().sum()
missing_final

CarNumber    0
Refund       0
Fines        0
Make         0
Model        9
dtype: int64

## 5. Split Make_n_model into Make and Model, drop it, and save to JSON

In [66]:
# Split Make_n_model into Make and Model
df["Make"] = df["Make_n_model"].apply(
    lambda x: str(x).split()[0] if pd.notna(x) else None
)

df["Model"] = df["Make_n_model"].apply(
    lambda x: " ".join(str(x).split()[1:])
    if pd.notna(x) and len(str(x).split()) > 1
    else None
)

# Drop original Make_n_model column
df = df.drop(columns=["Make_n_model"])

df[["CarNumber", "Make", "Model"]].head()

,CarNumber,Make,Model
ID,,,
0,Y163O8161RUS,Ford,Focus
1,E432XX77RUS,Toyota,Camry
2,7184TT36RUS,Ford,Focus
3,X582HE161RUS,Ford,Focus
5,92918M178RUS,Ford,Focus


## save to JSON

In [69]:
# Save the final dataframe to JSON in records format
df.to_json("data/auto.json", orient="records")